In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 0 — Install ONNX Runtime + TF 2.20
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys, os
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def find_wheel(pattern):
    for p in INPUT_ROOT.rglob(pattern):
        return p
    raise FileNotFoundError(pattern)

ONNX_WHL = Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/"
                "onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl")
if ONNX_WHL.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    str(ONNX_WHL)], check=True)
    print("✅ ONNX Runtime installed")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                str(find_wheel("tensorboard-2.20.0-*.whl"))], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                str(find_wheel("tensorflow-2.20.0-*.whl"))], check=True)
print("✅ TF 2.20 installed")

try:
    import onnxruntime as ort
    _ONNX_AVAILABLE = True
    print("✅ ONNX Runtime available")
except ImportError:
    _ONNX_AVAILABLE = False
    print("⚠️  ONNX not available, falling back to TF SavedModel")

✅ ONNX Runtime installed
✅ TF 2.20 installed
✅ ONNX Runtime available


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — Mode switch  ("submit" for Kaggle rerun, "train" for local CV)
# ═══════════════════════════════════════════════════════════════════════
MODE = "submit"   # ← change to "train" for local CV
assert MODE in {"train", "submit"}
print(f"MODE = {MODE}")

MODE = submit


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — Imports, paths, master config
# ═══════════════════════════════════════════════════════════════════════
import os, re, gc, time, warnings, random
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

tf.experimental.numpy.experimental_enable_numpy_behavior()
try:
    tf.config.set_visible_devices([], "GPU")
except Exception:
    pass

# ── Reproducibility ────────────────────────────────────────────────────
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
_WALL_START = time.time()

# ── Paths ──────────────────────────────────────────────────────────────
BASE      = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/"
                 "tensorflow2/perch_v2_cpu/1")
WORK_DIR  = Path("/kaggle/working/cache")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ── Audio constants ────────────────────────────────────────────────────
SR             = 32_000
WINDOW_SEC     = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES   = 60 * SR
N_WINDOWS      = 12   # 12 × 5s = 60s

# ═══════════════════════════════════════════════════════════════════════
# MASTER CONFIG — all tunable hyperparameters in one place
# ═══════════════════════════════════════════════════════════════════════
CFG = {
    # ── I/O ─────────────────────────────────────────────────────────
    "batch_files":   16,
    "dryrun_n_files": 20 if MODE == "train" else 0,
    "run_oof":        MODE == "train",
    "verbose":        MODE == "train",

    # ── ProtoSSM ────────────────────────────────────────────────────
    "proto": {
        "d_model":          128,
        "d_state":          16,
        "n_ssm_layers":     2,
        "dropout":          0.15,
        "cross_attn_heads": 2,
        "n_sites":          20,
        "meta_dim":         16,
        "n_epochs":         80  if MODE == "train" else 45,
        "patience":         20  if MODE == "train" else 10,
        "lr":               8e-4,
        "weight_decay":     1e-3,
        "pos_weight_cap":   25.0,
        "distill_weight":   0.15,   # BCE + 0.15 * MSE(out, perch_logits)
        "swa_start_frac":   0.65,
        "swa_lr":           4e-4,
        "tta_shifts":       [0, 1, -1, 2, -2, 3, -3],  # 7 shifts (was 5)
    },

    # ── ResidualSSM ─────────────────────────────────────────────────
    "residual": {
        "d_model":           64,
        "d_state":           8,
        "n_epochs":          40  if MODE == "train" else 30,
        "patience":          12  if MODE == "train" else 8,
        "lr":                8e-4,
        "correction_weight": 0.30,
    },

    # ── MLP probes ──────────────────────────────────────────────────
    "mlp": {
        "pca_dim":     64,
        "hidden":      (128, 64),
        "max_iter":    300,
        "min_pos":     5,
        "alpha_blend": 0.4,
        "lr":          5e-4,
        "alpha_reg":   0.005,
        "patience":    15,
    },

    # ── Post-processing ─────────────────────────────────────────────
    "pp": {
        "prior_lambda":         0.4,
        "conf_scale_top_k":     2,
        "conf_scale_power":     0.4,
        "rank_scale_power":     0.4,
        "delta_smooth_alpha":   0.20,
        "n_windows":            N_WINDOWS,
    },

    # ── Blend weights ───────────────────────────────────────────────
    "blend": {
        "sed_w":    0.40,
        "v5_w":     0.15,
        "clap_w":   0.10,
        # gate thresholds
        "fake_only_thr":    0.50,
        "sed_low_thr":      0.05,
        "fake_blend":       0.08,
        "cont_radius":      3,
        "cont_df":          2.0,
        "cont_scale":       1.20,
        "cont_rank_thr":    0.88,
        "cont_local_thr":   0.75,
        "sed_cont_low":     0.12,
        "cont_blend":       0.15,
        "sed_only_thr":     0.95,
        "fake_rank_low":    0.80,
        "sed_only_blend":   0.12,
    },
}

print("✅ Config loaded")
print(f"   mode={MODE}  proto_epochs={CFG['proto']['n_epochs']}  "
      f"tta_shifts={len(CFG['proto']['tta_shifts'])}")

✅ Config loaded
   mode=submit  proto_epochs=45  tta_shifts=7


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — Data loading & label parsing
# ═══════════════════════════════════════════════════════════════════════
taxonomy          = pd.read_csv(BASE / "taxonomy.csv")
sample_sub        = pd.read_csv(BASE / "sample_submission.csv")
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}

FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m:
        return {"site": "unknown", "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}

def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(";"):
                t = t.strip()
                if t:
                    out.add(t)
    return sorted(out)

sc = (soundscape_labels
      .groupby(["filename", "start", "end"])["primary_label"]
      .apply(union_labels)
      .reset_index(name="label_list"))

sc["end_sec"] = pd.to_timedelta(sc["end"]).dt.total_seconds().astype(int)
sc["row_id"]  = (sc["filename"].str.replace(".ogg", "", regex=False)
                 + "_" + sc["end_sec"].astype(str))

_meta = sc["filename"].apply(parse_fname).apply(pd.Series)
sc    = pd.concat([sc, _meta], axis=1)

Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
for i, lbls in enumerate(sc["label_list"]):
    for lbl in lbls:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1

windows_per_file  = sc.groupby("filename").size()
full_files        = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc["fully_labeled"] = sc["filename"].isin(full_files)

full_rows = (sc[sc["fully_labeled"]]
             .sort_values(["filename", "end_sec"])
             .reset_index(drop=False))
Y_FULL = Y_SC[full_rows["index"].to_numpy()]

CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()

print(f"Classes: {N_CLASSES} | Fully-labeled files: {len(full_files)}")
print(f"Full-file windows: {len(full_rows)} | Active classes: {int((Y_FULL.sum(0) > 0).sum())}")

Classes: 234 | Fully-labeled files: 59
Full-file windows: 708 | Active classes: 71


In [5]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — Load Perch model (ONNX preferred, 150x faster)
# ═══════════════════════════════════════════════════════════════════════
import re as _re

birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn       = birdclassifier.signatures["serving_default"]

ONNX_PERCH_PATH = Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx")
USE_ONNX = _ONNX_AVAILABLE and ONNX_PERCH_PATH.exists()

if USE_ONNX:
    _so = ort.SessionOptions()
    _so.intra_op_num_threads = 4
    ONNX_SESSION    = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=_so,
                                            providers=["CPUExecutionProvider"])
    ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
    ONNX_OUT_MAP    = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
    print("✅ Using ONNX Perch (150x faster)")
else:
    print("⚠️  Using TF SavedModel Perch (slower)")

# ── Species mapping: competition labels → Perch vocabulary ────────────
bc_labels = (pd.read_csv(MODEL_DIR / "assets" / "labels.csv")
             .reset_index()
             .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"}))
NO_LABEL = len(bc_labels)

mapping = (taxonomy
           .merge(bc_labels, on="scientific_name", how="left"))
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL).astype(int)
lbl2bc  = mapping.set_index("primary_label")["bc_index"]

BC_INDICES    = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK   = BC_INDICES != NO_LABEL
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)

# ── Genus-proxy for unmapped species (keep biologically meaningful taxa) ─
UNMAPPED_POS  = np.where(~MAPPED_MASK)[0].astype(np.int32)
PROXY_TAXA    = {"Amphibia", "Insecta", "Aves"}
proxy_map     = {}

unmapped_df = taxonomy[taxonomy["primary_label"]
                       .isin([PRIMARY_LABELS[i] for i in UNMAPPED_POS])].copy()

for _, row in unmapped_df.iterrows():
    target = row["primary_label"]
    genus  = str(row["scientific_name"]).split()[0]
    hits   = bc_labels[
        bc_labels["scientific_name"]
        .astype(str)
        .str.match(_re.compile(rf"^{_re.escape(genus)}\s"), na=False)
    ]
    if len(hits) > 0 and CLASS_NAME_MAP.get(target) in PROXY_TAXA:
        proxy_map[label_to_idx[target]] = hits["bc_index"].astype(int).tolist()

print(f"Mapped: {MAPPED_MASK.sum()} / {N_CLASSES} | Genus-proxy: {len(proxy_map)} | Unmapped: {len(UNMAPPED_POS) - len(proxy_map)}")

✅ Using ONNX Perch (150x faster)
Mapped: 203 / 234 | Genus-proxy: 3 | Unmapped: 28


In [6]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — Perch inference engine (ONNX + multithreaded I/O prefetch)
# ═══════════════════════════════════════════════════════════════════════
import concurrent.futures

def read_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    else:
        y = y[:FILE_SAMPLES]
    return y

def run_perch(paths, batch_files=16, verbose=True):
    paths  = [Path(p) for p in paths]
    n_rows = len(paths) * N_WINDOWS

    row_ids   = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites     = np.empty(n_rows, dtype=object)
    hours     = np.zeros(n_rows, dtype=np.int16)
    scores    = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embs      = np.zeros((n_rows, 1536),      dtype=np.float32)

    wr  = 0
    itr = (tqdm(range(0, len(paths), batch_files), desc="Perch")
           if verbose else range(0, len(paths), batch_files))

    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as io_exec:
        next_paths   = paths[0:batch_files]
        future_audio = [io_exec.submit(read_60s, p) for p in next_paths]

        for start in itr:
            batch_paths  = next_paths
            batch_n      = len(batch_paths)
            batch_audio  = [f.result() for f in future_audio]

            # Prefetch next batch immediately
            next_start = start + batch_files
            if next_start < len(paths):
                next_paths   = paths[next_start:next_start + batch_files]
                future_audio = [io_exec.submit(read_60s, p) for p in next_paths]

            x  = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
            br = wr

            for bi, path in enumerate(batch_paths):
                y    = batch_audio[bi]
                meta = parse_fname(path.name)
                stem = path.stem
                x[bi * N_WINDOWS:(bi + 1) * N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
                row_ids  [wr:wr + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
                filenames[wr:wr + N_WINDOWS] = path.name
                sites    [wr:wr + N_WINDOWS] = meta["site"]
                hours    [wr:wr + N_WINDOWS] = meta["hour_utc"]
                wr += N_WINDOWS

            if USE_ONNX:
                outs   = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
                logits = outs[ONNX_OUT_MAP["label"]].astype(np.float32)
                emb    = outs[ONNX_OUT_MAP["embedding"]].astype(np.float32)
            else:
                out    = infer_fn(inputs=tf.convert_to_tensor(x))
                logits = out["label"].numpy().astype(np.float32)
                emb    = out["embedding"].numpy().astype(np.float32)

            scores[br:wr, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
            embs  [br:wr]             = emb

            for pos_idx, bc_idxs in proxy_map.items():
                bc_arr = np.array(bc_idxs, dtype=np.int32)
                scores[br:wr, pos_idx] = logits[:, bc_arr].max(axis=1)

            del x, logits, emb, batch_audio
            gc.collect()

    meta_df = pd.DataFrame({"row_id": row_ids, "filename": filenames,
                             "site": sites, "hour_utc": hours})
    return meta_df, scores, embs

print("✅ Perch inference engine defined")

✅ Perch inference engine defined


In [7]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — Build-or-load Perch training cache
# ═══════════════════════════════════════════════════════════════════════
EXTERNAL_CACHE_DIRS = [
    Path("/kaggle/input/notebooks/vyankteshdwivedi/notebook1b25083f0d"),
    Path("/kaggle/input/datasets/jaejohn/perch-meta"),
]
CACHE_META_LOCAL = WORK_DIR / "perch_meta.parquet"
CACHE_NPZ_LOCAL  = WORK_DIR / "perch_arrays.npz"

SCORE_KEYS = ["scores", "sc", "logits", "perch_scores", "preds", "arr_0"]
EMB_KEYS   = ["embs", "emb", "embeddings", "features", "perch_embs", "arr_1"]

def _pick_array(arr, candidates, shape_hint_cols):
    for k in candidates:
        if k in arr.files:
            return arr[k], k
    for k in arr.files:
        v = arr[k]
        if v.ndim == 2 and v.shape[1] == shape_hint_cols:
            return v, k
    raise KeyError(f"None of {candidates} found. Available: {arr.files}")

def _find_external_cache():
    for d in EXTERNAL_CACHE_DIRS:
        m, n = d / "perch_meta.parquet", d / "perch_arrays.npz"
        if m.exists() and n.exists():
            return m, n
    return None, None

def _build_cache():
    print(f"Building Perch cache from {len(full_files)} training files…")
    train_paths = [p for p in [BASE / "train_soundscapes" / fn for fn in full_files] if p.exists()]
    t0 = time.time()
    meta_b, sc_b, emb_b = run_perch(train_paths, CFG["batch_files"], verbose=True)
    print(f"  Perch done in {time.time()-t0:.1f}s  scores={sc_b.shape} embs={emb_b.shape}")
    meta_b.to_parquet(CACHE_META_LOCAL)
    np.savez(CACHE_NPZ_LOCAL,
             scores=sc_b.astype(np.float32),
             embs=emb_b.astype(np.float32),
             primary_labels=np.array(PRIMARY_LABELS))
    print(f"  Cache saved → {WORK_DIR}")
    return CACHE_META_LOCAL, CACHE_NPZ_LOCAL

ext_meta, ext_npz = _find_external_cache()
if ext_meta:
    CACHE_META, CACHE_NPZ = ext_meta, ext_npz
    print(f"Using external cache: {CACHE_META.parent}")
elif CACHE_META_LOCAL.exists() and CACHE_NPZ_LOCAL.exists():
    CACHE_META, CACHE_NPZ = CACHE_META_LOCAL, CACHE_NPZ_LOCAL
    print(f"Using local cache: {WORK_DIR}")
else:
    print("No cache — building from scratch (~1.5 min)")
    CACHE_META, CACHE_NPZ = _build_cache()

meta_tr = pd.read_parquet(CACHE_META)
_arr    = np.load(CACHE_NPZ)
sc_tr_raw,  sk = _pick_array(_arr, SCORE_KEYS, N_CLASSES)
emb_tr_raw, ek = _pick_array(_arr, EMB_KEYS,   1536)
sc_tr  = sc_tr_raw.astype(np.float32)
emb_tr = emb_tr_raw.astype(np.float32)

if "primary_labels" in _arr.files:
    assert _arr["primary_labels"].tolist() == PRIMARY_LABELS, "Cache label mismatch!"

if "row_id" not in meta_tr.columns:
    end_sec = np.tile(np.arange(5, 65, 5), len(meta_tr) // N_WINDOWS)
    meta_tr["row_id"] = (meta_tr["filename"].str.replace(".ogg", "", regex=False)
                          + "_" + pd.Series(end_sec).astype(str).values)

row_id_to_index = full_rows.set_index("row_id")["index"]
missing = set(meta_tr["row_id"]) - set(row_id_to_index.index)
if missing:
    raise RuntimeError(f"Cache has {len(missing)} unknown row_ids. Delete cache and rebuild.")

Y_FULL_aligned = Y_SC[row_id_to_index.loc[meta_tr["row_id"]].to_numpy()]

print(f"✅ sc_tr={sc_tr.shape}  emb_tr={emb_tr.shape}  Y_FULL_aligned={Y_FULL_aligned.shape}")

No cache — building from scratch (~1.5 min)
Building Perch cache from 59 training files…


Perch:   0%|          | 0/4 [00:00<?, ?it/s]

  Perch done in 141.9s  scores=(708, 234) embs=(708, 1536)
  Cache saved → /kaggle/working/cache
✅ sc_tr=(708, 234)  emb_tr=(708, 1536)  Y_FULL_aligned=(708, 234)


In [8]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 — Validation helpers
# ═══════════════════════════════════════════════════════════════════════
def macro_auc(y_true, y_score):
    """Competition metric: macro-AUC skipping absent classes."""
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")

def honest_oof_auc(scores, Y, meta_df, n_splits=5, label="scores"):
    """GroupKFold by filename — the ONLY correct local estimate."""
    groups = meta_df["filename"].to_numpy()
    gkf    = GroupKFold(n_splits=n_splits)
    oof    = np.zeros_like(scores, dtype=np.float32)
    for _, (_, va_idx) in enumerate(gkf.split(scores, groups=groups), 1):
        oof[va_idx] = scores[va_idx]
    auc = macro_auc(Y, oof)
    print(f"[{label}] honest OOF macro-AUC: {auc:.6f}")
    return auc, oof

if CFG["run_oof"]:
    baseline_auc, _ = honest_oof_auc(sc_tr, Y_FULL_aligned, meta_tr,
                                      n_splits=5, label="raw Perch")
    print(f"Baseline OOF AUC: {baseline_auc:.6f}")
else:
    print("Submit mode: skipping raw OOF")

Submit mode: skipping raw OOF


In [9]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8b — Full pipeline OOF (train mode only)
# GroupKFold by filename → trains ProtoSSM + MLP on K-1 folds,
# predicts on held-out fold. Use this for reliable local CV.
# ═══════════════════════════════════════════════════════════════════════
def run_pipeline_oof(emb_full, sc_full, Y_full, meta_full, n_splits=5):
    """
    Proper full-pipeline OOF: trains ProtoSSM + MLP on K-1 folds.
    GroupKFold by filename — the ONLY correct local estimate.
    ~4-6 min total on CPU.
    """
    file_meta = meta_full.drop_duplicates("filename").reset_index(drop=True)
    gkf       = GroupKFold(n_splits=n_splits)
    oof_probs = np.zeros((len(sc_full), N_CLASSES), dtype=np.float32)

    for fold, (tr_f, va_f) in enumerate(
            gkf.split(file_meta, groups=file_meta["filename"]), 1):

        tr_fnames = set(file_meta.iloc[tr_f]["filename"])
        va_fnames = set(file_meta.iloc[va_f]["filename"])
        tr_mask   = meta_full["filename"].isin(tr_fnames).values
        va_mask   = meta_full["filename"].isin(va_fnames).values

        emb_tr_f  = emb_full[tr_mask];  sc_tr_f = sc_full[tr_mask]
        Y_tr_f    = Y_full[tr_mask];    meta_tr_f = meta_full[tr_mask].reset_index(drop=True)
        emb_va_f  = emb_full[va_mask];  sc_va_f = sc_full[va_mask]
        meta_va_f = meta_full[va_mask].reset_index(drop=True)

        # ── Train ProtoSSM on train fold ──────────────────────────────
        proto_m, site2i_f = train_light_proto_ssm(
            emb_tr_f, sc_tr_f, Y_tr_f, meta_tr_f,
            n_epochs=CFG["proto"]["n_epochs"],
            patience=CFG["proto"]["patience"],
            lr=CFG["proto"]["lr"],
            n_sites=CFG["proto"]["n_sites"],
            verbose=False)

        # ── ProtoSSM predict on val fold ──────────────────────────────
        n_va = len(emb_va_f) // N_WINDOWS
        va_fn_list = meta_va_f.drop_duplicates("filename")["filename"].tolist()
        va_site_ids = np.array([
            min(site2i_f.get(meta_va_f.loc[meta_va_f["filename"]==fn,"site"].iloc[0], 0), 19)
            for fn in va_fn_list], dtype=np.int64)
        va_hour_ids = np.array([
            int(meta_va_f.loc[meta_va_f["filename"]==fn,"hour_utc"].iloc[0]) % 24
            for fn in va_fn_list], dtype=np.int64)

        proto_m.eval()
        with torch.no_grad():
            proto_va = proto_m(
                torch.tensor(emb_va_f.reshape(n_va, N_WINDOWS, -1), dtype=torch.float32),
                torch.tensor(sc_va_f.reshape(n_va, N_WINDOWS, -1),  dtype=torch.float32),
                site_ids=torch.tensor(va_site_ids, dtype=torch.long),
                hours   =torch.tensor(va_hour_ids, dtype=torch.long),
            ).numpy().reshape(-1, N_CLASSES)

        # ── MLP probes on train fold ───────────────────────────────────
        probe_m, scaler_f, pca_f, alpha_b = train_mlp_probes(
            emb_tr_f, sc_tr_f, Y_tr_f,
            min_pos=CFG["mlp"]["min_pos"],
            pca_dim=CFG["mlp"]["pca_dim"],
            alpha_blend=CFG["mlp"]["alpha_blend"])
        sc_va_mlp = apply_mlp_probes_vectorized(
            emb_va_f, sc_va_f, probe_m, scaler_f, pca_f, alpha_b)

        # ── Prior + first-pass ensemble ────────────────────────────────
        pt = build_prior_tables(meta_va_f, Y_full[va_mask])  # val-only prior (conservative)
        sc_va_adj = apply_prior(sc_va_f,
                                sites=meta_va_f["site"].to_numpy(),
                                hours=meta_va_f["hour_utc"].to_numpy(),
                                tables=pt,
                                lambda_prior=CFG["pp"]["prior_lambda"])

        first_pass = 0.5 * proto_va + 0.5 * sc_va_mlp
        probs_va   = sigmoid(first_pass)
        oof_probs[va_mask] = probs_va

        fold_auc = macro_auc(Y_full[va_mask], probs_va)
        print(f"  Fold {fold}/{n_splits}  val_files={len(va_fnames)}  AUC={fold_auc:.6f}")

        del proto_m, probe_m; gc.collect()

    overall = macro_auc(Y_full, oof_probs)
    print(f"\nFull pipeline OOF AUC: {overall:.6f}")
    return overall, oof_probs


if CFG["run_oof"]:
    print("Running full-pipeline OOF (train mode)…")
    oof_auc, oof_pipeline = run_pipeline_oof(
        emb_tr, sc_tr, Y_FULL_aligned, meta_tr,
        n_splits=5)
else:
    print("Submit mode: skipping full OOF")


Submit mode: skipping full OOF


In [10]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8 — Post-processing helpers (all 7 post-proc steps)
# ═══════════════════════════════════════════════════════════════════════

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

# ── 8a: Site/Hour/SiteHour 3-tier prior (NB1 innovation) ──────────────
def build_prior_tables(sc_df, Y_labels):
    """Build global, site, hour, AND joint site×hour frequency tables."""
    sc_df    = sc_df.reset_index(drop=True)
    global_p = Y_labels.mean(axis=0).astype(np.float32)

    def _build_level(keys, get_key_fn):
        k2i = {k: i for i, k in enumerate(keys)}
        p   = np.zeros((len(keys), Y_labels.shape[1]), dtype=np.float32)
        n   = np.zeros(len(keys), dtype=np.float32)
        for k in keys:
            mask = np.array([get_key_fn(row) == k for _, row in sc_df.iterrows()], dtype=bool)
            n[k2i[k]] = mask.sum()
            if mask.sum() > 0:
                p[k2i[k]] = Y_labels[mask].mean(axis=0)
        return k2i, p, n

    site_keys = sorted(sc_df["site"].dropna().astype(str).unique())
    site2i, site_p, site_n = {}, np.zeros((len(site_keys), Y_labels.shape[1]), np.float32), np.zeros(len(site_keys), np.float32)
    for i, s in enumerate(site_keys):
        site2i[s] = i
        mask = sc_df["site"].astype(str).values == s
        site_n[i] = mask.sum()
        if mask.sum() > 0: site_p[i] = Y_labels[mask].mean(0)

    hour_keys = sorted(sc_df["hour_utc"].dropna().astype(int).unique())
    hour2i, hour_p, hour_n = {}, np.zeros((len(hour_keys), Y_labels.shape[1]), np.float32), np.zeros(len(hour_keys), np.float32)
    for i, h in enumerate(hour_keys):
        hour2i[h] = i
        mask = sc_df["hour_utc"].astype(int).values == h
        hour_n[i] = mask.sum()
        if mask.sum() > 0: hour_p[i] = Y_labels[mask].mean(0)

    # ── Joint site×hour (NB1 unique) ───────────────────────────────────
    sh_keys = sorted({(str(s), int(h))
                      for s, h in zip(sc_df["site"].dropna(), sc_df["hour_utc"].dropna())})
    sh2i, sh_p, sh_n = {}, np.zeros((len(sh_keys), Y_labels.shape[1]), np.float32), np.zeros(len(sh_keys), np.float32)
    for i, (s, h) in enumerate(sh_keys):
        sh2i[(s, h)] = i
        mask = (sc_df["site"].astype(str).values == s) & (sc_df["hour_utc"].astype(int).values == h)
        sh_n[i] = mask.sum()
        if mask.sum() > 0: sh_p[i] = Y_labels[mask].mean(0)

    return dict(global_p=global_p,
                site2i=site2i, site_p=site_p, site_n=site_n,
                hour2i=hour2i, hour_p=hour_p, hour_n=hour_n,
                sh2i=sh2i, sh_p=sh_p, sh_n=sh_n)

def apply_prior(scores, sites, hours, tables, lambda_prior=0.4):
    eps = 1e-4
    n   = len(scores)
    out = scores.copy()
    p   = np.tile(tables["global_p"], (n, 1))

    for i, h in enumerate(hours):
        h = int(h)
        if h in tables["hour2i"]:
            j = tables["hour2i"][h]; nh = tables["hour_n"][j]
            w = nh / (nh + 8.0)
            p[i] = w * tables["hour_p"][j] + (1 - w) * tables["global_p"]

    for i, s in enumerate(sites):
        s = str(s)
        if s in tables["site2i"]:
            j = tables["site2i"][s]; ns = tables["site_n"][j]
            w = ns / (ns + 8.0)
            p[i] = w * tables["site_p"][j] + (1 - w) * p[i]

    for i, (s, h) in enumerate(zip(sites, hours)):
        key = (str(s), int(h))
        if key in tables["sh2i"]:
            j = tables["sh2i"][key]; nsh = tables["sh_n"][j]
            w = nsh / (nsh + 4.0)
            p[i] = w * tables["sh_p"][j] + (1 - w) * p[i]

    p    = np.clip(p, eps, 1 - eps)
    out += lambda_prior * (np.log(p) - np.log1p(-p))
    return out.astype(np.float32)

# ── 8b: File confidence scaling ────────────────────────────────────────
def file_confidence_scale(probs, n_windows=N_WINDOWS, top_k=2, power=0.4):
    N, C = probs.shape
    view = probs.reshape(-1, n_windows, C)
    top_k_mean = np.sort(view, axis=1)[:, -top_k:, :].mean(axis=1, keepdims=True)
    return (view * np.power(top_k_mean, power)).reshape(N, C)

# ── 8c: Rank-aware scaling ─────────────────────────────────────────────
def rank_aware_scaling(probs, n_windows=N_WINDOWS, power=0.4):
    N, C = probs.shape
    view = probs.reshape(-1, n_windows, C)
    file_max = view.max(axis=1, keepdims=True)
    return (view * np.power(file_max, power)).reshape(N, C)

# ── 8d: Adaptive delta smoothing ──────────────────────────────────────
def adaptive_delta_smooth(probs, n_windows=N_WINDOWS, base_alpha=0.20):
    N, C   = probs.shape
    result = probs.copy()
    view   = probs.reshape(-1, n_windows, C)
    out    = result.reshape(-1, n_windows, C)
    for t in range(n_windows):
        conf   = view[:, t, :].max(axis=-1, keepdims=True)
        alpha  = base_alpha * (1.0 - conf)
        if t == 0:
            nbr = (view[:, t, :] + view[:, t+1, :]) / 2.0
        elif t == n_windows - 1:
            nbr = (view[:, t-1, :] + view[:, t, :]) / 2.0
        else:
            nbr = (view[:, t-1, :] + view[:, t+1, :]) / 2.0
        out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * nbr
    return result

# ── 8e: Per-taxon temperature scaling ─────────────────────────────────
TEXTURE_TAXA = {"Amphibia", "Insecta"}
temperatures = np.ones(N_CLASSES, dtype=np.float32)
for ci, label in enumerate(PRIMARY_LABELS):
    cls = CLASS_NAME_MAP.get(label, "Aves")
    temperatures[ci] = 0.95 if cls in TEXTURE_TAXA else 1.10

print(f"✅ Post-processing helpers defined | Temperatures: {(temperatures < 1).sum()} texture / {(temperatures > 1).sum()} event")

✅ Post-processing helpers defined | Temperatures: 63 texture / 171 event


In [11]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 9 — Isotonic calibration + per-class threshold optimization
# ═══════════════════════════════════════════════════════════════════════
from sklearn.isotonic import IsotonicRegression

def calibrate_and_optimize_thresholds(oof_probs, Y_FULL,
                                       threshold_grid=None, n_windows=N_WINDOWS):
    """Isotonic regression + F1-optimal threshold per species."""
    if threshold_grid is None:
        threshold_grid = np.arange(0.20, 0.75, 0.05).tolist()

    n_samples, n_cls = oof_probs.shape
    thresholds       = np.full(n_cls, 0.5, dtype=np.float32)
    n_files          = n_samples // n_windows
    file_oof         = oof_probs.reshape(n_files, n_windows, n_cls).max(axis=1)
    file_y           = Y_FULL.reshape(n_files, n_windows, n_cls).max(axis=1)

    n_calibrated = 0
    for c in range(n_cls):
        y_true, y_prob = file_y[:, c], file_oof[:, c]
        if y_true.sum() < 3:
            continue
        try:
            ir    = IsotonicRegression(out_of_bounds="clip")
            ir.fit(y_prob, y_true)
            y_cal = ir.transform(y_prob)
        except Exception:
            y_cal = y_prob

        best_f1, best_t = 0.0, 0.5
        for t in threshold_grid:
            pred = (y_cal >= t).astype(int)
            tp   = ((pred==1) & (y_true==1)).sum()
            fp   = ((pred==1) & (y_true==0)).sum()
            fn   = ((pred==0) & (y_true==1)).sum()
            prec = tp / (tp + fp + 1e-8)
            rec  = tp / (tp + fn + 1e-8)
            f1   = 2 * prec * rec / (prec + rec + 1e-8)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds[c] = best_t
        n_calibrated += 1

    print(f"Calibrated {n_calibrated}/{n_cls} classes | "
          f"mean_thr={thresholds.mean():.3f} "
          f"range=[{thresholds.min():.2f}, {thresholds.max():.2f}]")
    return thresholds

def apply_per_class_thresholds(scores, thresholds):
    scaled = np.copy(scores)
    for c in range(scores.shape[1]):
        t     = thresholds[c]
        above = scores[:, c] > t
        scaled[ above, c] = 0.5 + 0.5 * (scores[ above, c] - t) / (1 - t + 1e-8)
        scaled[~above, c] = 0.5 * scores[~above, c] / (t + 1e-8)
    return np.clip(scaled, 0.0, 1.0)

print("✅ Isotonic calibration + threshold optimization defined")

✅ Isotonic calibration + threshold optimization defined


In [12]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 10 — Vectorized MLP probes on PCA-64 embeddings
# ═══════════════════════════════════════════════════════════════════════
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

def build_class_freq_weights(Y, cap=10.0):
    freq    = (Y.sum(axis=0).astype(np.float32) + 1.0) / Y.shape[0]
    weights = np.clip(1.0 / (freq ** 0.5), 1.0, cap)
    return (weights / weights.mean()).astype(np.float32)

def build_sequential_features(scores_col, n_windows=N_WINDOWS):
    x     = scores_col.reshape(-1, n_windows)
    prev  = np.concatenate([x[:, :1],  x[:, :-1]], axis=1)
    next_ = np.concatenate([x[:, 1:],  x[:, -1:]], axis=1)
    mean  = np.repeat(x.mean(axis=1), n_windows)
    max_  = np.repeat(x.max(axis=1),  n_windows)
    std   = np.repeat(x.std(axis=1),  n_windows)
    return prev.reshape(-1), next_.reshape(-1), mean, max_, std

def train_mlp_probes(emb, scores_raw, Y, min_pos=5, pca_dim=64, alpha_blend=0.4):
    scaler = StandardScaler()
    emb_s  = scaler.fit_transform(emb)
    pca    = PCA(n_components=min(pca_dim, emb_s.shape[1] - 1))
    Z      = pca.fit_transform(emb_s).astype(np.float32)
    print(f"  PCA: {emb.shape} → {Z.shape}  var={pca.explained_variance_ratio_.sum():.2%}")

    class_weights = build_class_freq_weights(Y)
    probe_models  = {}
    active        = np.where(Y.sum(axis=0) >= min_pos)[0]
    MAX_ROWS      = 3000

    for ci in tqdm(active, desc="MLP probes", leave=False):
        y = Y[:, ci]
        if y.sum() == 0 or y.sum() == len(y):
            continue
        prev, next_, mean, max_, std = build_sequential_features(scores_raw[:, ci])
        X = np.hstack([Z, scores_raw[:, ci:ci+1],
                       prev[:, None], next_[:, None],
                       mean[:, None], max_[:, None], std[:, None]])
        n_pos = int(y.sum())
        pos_idx = np.where(y == 1)[0]
        w = float(class_weights[ci])
        repeat = min(max(1, int(round(w * (len(y) - n_pos) / max(n_pos, 1)))), 8)
        if n_pos * repeat + len(y) > MAX_ROWS:
            repeat = max(1, (MAX_ROWS - len(y)) // max(n_pos, 1))
        X_bal = np.vstack([X, np.tile(X[pos_idx], (repeat, 1))])
        y_bal = np.concatenate([y, np.ones(n_pos * repeat, dtype=y.dtype)])
        clf = MLPClassifier(
            hidden_layer_sizes=CFG["mlp"]["hidden"],
            activation="relu",
            max_iter=CFG["mlp"]["max_iter"],
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=CFG["mlp"]["patience"],
            random_state=42,
            learning_rate_init=CFG["mlp"]["lr"],
            alpha=CFG["mlp"]["alpha_reg"],
        )
        clf.fit(X_bal, y_bal)
        probe_models[ci] = clf

    print(f"  Trained {len(probe_models)} MLP probes")
    return probe_models, scaler, pca, alpha_blend


class VectorizedMLPProbes(nn.Module):
    """Stacks all per-class sklearn MLP weights into a single batched PyTorch model."""
    def __init__(self, probe_models):
        super().__init__()
        self.valid_classes = sorted(probe_models.keys())
        V = len(self.valid_classes)
        if V == 0:
            self.n_layers = 0
            self.weights = nn.ParameterList()
            self.biases  = nn.ParameterList()
            return
        sample = probe_models[self.valid_classes[0]]
        self.n_layers = len(sample.coefs_)
        self.weights, self.biases = nn.ParameterList(), nn.ParameterList()
        for li in range(self.n_layers):
            W = np.stack([probe_models[c].coefs_[li] for c in self.valid_classes], axis=0)
            b = np.stack([probe_models[c].intercepts_[li] for c in self.valid_classes], axis=0)
            self.weights.append(nn.Parameter(torch.tensor(W, dtype=torch.float32), requires_grad=False))
            self.biases.append( nn.Parameter(torch.tensor(b, dtype=torch.float32), requires_grad=False))

    def forward(self, x):
        h = x
        for i in range(self.n_layers):
            h = torch.bmm(h, self.weights[i]) + self.biases[i].unsqueeze(1)
            if i < self.n_layers - 1:
                h = torch.relu(h)
        return h.squeeze(-1)   # (V, N)


def apply_mlp_probes_vectorized(emb_test, scores_test, probe_models,
                                  scaler, pca, alpha_blend=0.4):
    if len(probe_models) == 0:
        return scores_test.copy()
    emb_s  = scaler.transform(emb_test)
    Z_test = pca.transform(emb_s).astype(np.float32)
    valid_classes = sorted(probe_models.keys())
    V, N = len(valid_classes), len(scores_test)
    raw      = scores_test[:, valid_classes].T   # (V, N)
    n_files  = N // N_WINDOWS
    raw_view = raw.reshape(V, n_files, N_WINDOWS)
    prev = np.concatenate([raw_view[:, :, :1], raw_view[:, :, :-1]], axis=2).reshape(V, N)
    nxt  = np.concatenate([raw_view[:, :, 1:], raw_view[:, :, -1:]], axis=2).reshape(V, N)
    mean = np.repeat(raw_view.mean(axis=2), N_WINDOWS, axis=1)
    mx   = np.repeat(raw_view.max(axis=2),  N_WINDOWS, axis=1)
    std  = np.repeat(raw_view.std(axis=2),  N_WINDOWS, axis=1)
    scalar_feats = np.stack([raw, prev, nxt, mean, mx, std], axis=-1).astype(np.float32)
    Z_exp = np.broadcast_to(Z_test, (V, N, Z_test.shape[1]))
    X_all = np.concatenate([Z_exp.astype(np.float32), scalar_feats], axis=-1)
    vec_probe = VectorizedMLPProbes(probe_models)
    vec_probe.eval()
    with torch.no_grad():
        preds = vec_probe(torch.tensor(X_all)).numpy()  # (V, N)
    result = scores_test.copy()
    result[:, valid_classes] = ((1.0 - alpha_blend) * scores_test[:, valid_classes]
                                + alpha_blend * preds.T)
    return result

print("✅ Vectorized MLP probe (PCA-64, hidden=(128,64)) defined")

✅ Vectorized MLP probe (PCA-64, hidden=(128,64)) defined


In [13]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 11 — Selective SSM building block
# ═══════════════════════════════════════════════════════════════════════

class SelectiveSSM(nn.Module):
    """Mamba-inspired selective state space model (CPU-efficient, T=12)."""
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model, self.d_state = d_model, d_state
        self.in_proj  = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d   = nn.Conv1d(d_model, d_model, d_conv, padding=d_conv-1, groups=d_model)
        self.dt_proj  = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state+1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.A_log  = nn.Parameter(torch.log(A))
        self.D      = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B_sz, T, D = x.shape
        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :T].transpose(1, 2)
        x_conv = F.silu(x_conv)
        dt     = F.softplus(self.dt_proj(x_conv))
        A      = -torch.exp(self.A_log)
        B, C   = self.B_proj(x_conv), self.C_proj(x_conv)
        h      = torch.zeros(B_sz, D, self.d_state, device=x.device)
        ys     = []
        for t in range(T):
            dA = torch.exp(A[None] * dt[:, t, :, None])
            dB = dt[:, t, :, None] * B[:, t, None, :]
            h  = h * dA + x[:, t, :, None] * dB
            ys.append((h * C[:, t, None, :]).sum(-1))
        y = torch.stack(ys, dim=1)
        return y + x * self.D[None, None, :]

print("✅ SelectiveSSM building block defined")

✅ SelectiveSSM building block defined


In [14]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 12 — LightProtoSSM v2 (bi-SSM + cross-attention + SWA)
# ═══════════════════════════════════════════════════════════════════════

class LightProtoSSM(nn.Module):
    """
    Core sequence model.
    Architecture:
      input_proj (1536 → d_model) → positional encoding + site/hour meta
      → 2× [Bi-SSM block + LayerNorm + Cross-Attention + LayerNorm]
      → cosine-similarity to learnable prototypes
      → learnable fusion alpha with raw Perch logits
    """
    def __init__(self, d_input=1536, d_model=128, d_state=16,
                 n_classes=234, n_windows=12, dropout=0.15,
                 n_sites=20, meta_dim=16,
                 n_ssm_layers=2, cross_attn_heads=2):
        super().__init__()
        self.n_classes, self.n_windows = n_classes, n_windows

        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model),
            nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.pos_enc   = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.site_emb  = nn.Embedding(n_sites, meta_dim)
        self.hour_emb  = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)

        self.ssm_fwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(n_ssm_layers)])
        self.ssm_bwd   = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(n_ssm_layers)])
        self.ssm_merge = nn.ModuleList([nn.Linear(2*d_model, d_model)   for _ in range(n_ssm_layers)])
        self.ssm_norm  = nn.ModuleList([nn.LayerNorm(d_model)           for _ in range(n_ssm_layers)])
        self.drop      = nn.Dropout(dropout)
        self.cross_attn = nn.ModuleList([
            nn.MultiheadAttention(d_model, num_heads=cross_attn_heads,
                                   dropout=dropout, batch_first=True)
            for _ in range(n_ssm_layers)])
        self.cross_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_ssm_layers)])

        self.prototypes   = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp   = nn.Parameter(torch.tensor(5.0))
        self.class_bias   = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))

    def init_prototypes(self, emb_tensor, labels_tensor):
        with torch.no_grad():
            h = self.input_proj(emb_tensor)
            for c in range(self.n_classes):
                mask = labels_tensor[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, emb, perch_logits=None, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(emb) + self.pos_enc[:, :T, :]
        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat(
                [self.site_emb(site_ids), self.hour_emb(hours)], dim=-1))
            h = h + meta[:, None, :]

        for i in range(len(self.ssm_fwd)):
            res = h
            h_f = self.ssm_fwd[i](h)
            h_b = self.ssm_bwd[i](h.flip(1)).flip(1)
            h   = self.drop(self.ssm_merge[i](torch.cat([h_f, h_b], dim=-1)))
            h   = self.ssm_norm[i](h + res)
            attn_out, _ = self.cross_attn[i](h, h, h)
            h = self.cross_norm[i](h + attn_out)

        h_n = F.normalize(h, dim=-1)
        p_n = F.normalize(self.prototypes, dim=-1)
        sim = (torch.matmul(h_n, p_n.T) * F.softplus(self.proto_temp)
               + self.class_bias[None, None, :])

        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
            return alpha * sim + (1 - alpha) * perch_logits
        return sim


def train_light_proto_ssm(emb_full, scores_full, Y_full, meta_full,
                           n_epochs=45, patience=10, lr=8e-4,
                           n_sites=20, verbose=False):
    pcfg = CFG["proto"]
    n_files = len(emb_full) // N_WINDOWS
    emb_f   = emb_full.reshape(n_files, N_WINDOWS, -1)
    log_f   = scores_full.reshape(n_files, N_WINDOWS, -1)
    lab_f   = Y_full.reshape(n_files, N_WINDOWS, -1).astype(np.float32)

    fnames  = meta_full["filename"].unique()
    sites_u = sorted(meta_full["site"].unique())
    site2i  = {s: i + 1 for i, s in enumerate(sites_u)}

    site_ids = np.array([
        min(site2i.get(meta_full.loc[meta_full["filename"]==fn,"site"].iloc[0], 0), n_sites-1)
        for fn in fnames], dtype=np.int64)
    hour_ids = np.array([
        int(meta_full.loc[meta_full["filename"]==fn,"hour_utc"].iloc[0]) % 24
        for fn in fnames], dtype=np.int64)

    model = LightProtoSSM(
        n_classes=N_CLASSES, d_model=pcfg["d_model"], d_state=pcfg["d_state"],
        n_ssm_layers=pcfg["n_ssm_layers"], dropout=pcfg["dropout"],
        cross_attn_heads=pcfg["cross_attn_heads"],
        n_sites=n_sites, meta_dim=pcfg["meta_dim"],
    )
    model.init_prototypes(
        torch.tensor(emb_full, dtype=torch.float32),
        torch.tensor(Y_full, dtype=torch.float32))
    print(f"  LightProtoSSM params: {model.count_parameters():,}")

    emb_t  = torch.tensor(emb_f,    dtype=torch.float32)
    log_t  = torch.tensor(log_f,    dtype=torch.float32)
    lab_t  = torch.tensor(lab_f,    dtype=torch.float32)
    site_t = torch.tensor(site_ids, dtype=torch.long)
    hour_t = torch.tensor(hour_ids, dtype=torch.long)

    pos_cnt    = lab_t.sum(dim=(0, 1))
    total      = lab_t.shape[0] * lab_t.shape[1]
    pos_weight = ((total - pos_cnt) / (pos_cnt + 1)).clamp(max=pcfg["pos_weight_cap"])

    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=pcfg["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, epochs=n_epochs, steps_per_epoch=1, pct_start=0.1, anneal_strategy="cos")

    swa_model = torch.optim.swa_utils.AveragedModel(model)
    swa_start = int(n_epochs * pcfg["swa_start_frac"])
    swa_sched = torch.optim.swa_utils.SWALR(opt, swa_lr=pcfg["swa_lr"])

    best_loss, best_state, wait = float("inf"), None, 0

    for ep in range(n_epochs):
        model.train()
        out  = model(emb_t, log_t, site_ids=site_t, hours=hour_t)
        loss = (F.binary_cross_entropy_with_logits(
                    out, lab_t, pos_weight=pos_weight[None, None, :])
                + pcfg["distill_weight"] * F.mse_loss(out, log_t))
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if ep >= swa_start:
            swa_model.update_parameters(model)
            swa_sched.step()
        else:
            sched.step()

        if loss.item() < best_loss:
            best_loss  = loss.item()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            if verbose: print(f"  Early stop ep {ep+1}")
            break

    if ep >= swa_start:
        torch.optim.swa_utils.update_bn(emb_t.unsqueeze(0), swa_model)
        model = swa_model
    else:
        model.load_state_dict(best_state)

    model.eval()
    print(f"  ProtoSSM trained — best loss={best_loss:.4f}")
    return model, site2i


def run_tta_proto(proto_model, emb_files, sc_files, site_t, hour_t,
                  shifts=None):
    """7-shift circular TTA (extended from 5 in source notebooks)."""
    if shifts is None:
        shifts = CFG["proto"]["tta_shifts"]
    proto_model.eval()
    all_preds = []
    emb_t = torch.tensor(emb_files, dtype=torch.float32)
    sc_t  = torch.tensor(sc_files,  dtype=torch.float32)
    for shift in shifts:
        e_s = torch.roll(emb_t, shift, dims=1) if shift != 0 else emb_t
        s_s = torch.roll(sc_t,  shift, dims=1) if shift != 0 else sc_t
        with torch.no_grad():
            out = proto_model(e_s, s_s, site_ids=site_t, hours=hour_t).numpy()
        if shift != 0:
            out = np.roll(out, -shift, axis=1)
        all_preds.append(out)
    return np.mean(all_preds, axis=0)   # (n_files, 12, 234)


print("✅ LightProtoSSM v2 (bi-SSM + 2-head cross-attn + SWA) defined")

✅ LightProtoSSM v2 (bi-SSM + 2-head cross-attn + SWA) defined


In [15]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 13 — ResidualSSM (second-pass error correction, zero-init head)
# ═══════════════════════════════════════════════════════════════════════

class ResidualSSM(nn.Module):
    def __init__(self, d_input=1536, d_scores=234, d_model=64, d_state=8,
                 n_classes=234, n_windows=12, dropout=0.1, n_sites=20, meta_dim=8):
        super().__init__()
        self.n_classes = n_classes
        self.input_proj = nn.Sequential(
            nn.Linear(d_input + d_scores, d_model),
            nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.site_emb  = nn.Embedding(n_sites, meta_dim)
        self.hour_emb  = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.pos_enc   = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.ssm_fwd   = SelectiveSSM(d_model, d_state)
        self.ssm_bwd   = SelectiveSSM(d_model, d_state)
        self.ssm_merge = nn.Linear(2 * d_model, d_model)
        self.ssm_norm  = nn.LayerNorm(d_model)
        self.ssm_drop  = nn.Dropout(dropout)
        self.output_head = nn.Linear(d_model, n_classes)
        nn.init.zeros_(self.output_head.weight)  # zero init — corrections start at 0
        nn.init.zeros_(self.output_head.bias)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, emb, first_pass, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(torch.cat([emb, first_pass], dim=-1)) + self.pos_enc[:, :T, :]
        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat(
                [self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings-1)),
                 self.hour_emb(hours.clamp(0, 23))], dim=-1))
            h = h + meta.unsqueeze(1)
        res = h
        h_f = self.ssm_fwd(h)
        h_b = self.ssm_bwd(h.flip(1)).flip(1)
        h   = self.ssm_drop(self.ssm_merge(torch.cat([h_f, h_b], dim=-1)))
        h   = self.ssm_norm(h + res)
        return self.output_head(h)


def train_residual_ssm(emb_full, first_pass_flat, Y_full, site_ids, hour_ids,
                       n_epochs=30, patience=8, lr=8e-4, correction_weight=0.30,
                       verbose=False):
    rcfg = CFG["residual"]
    n_files   = len(emb_full) // N_WINDOWS
    emb_f     = emb_full.reshape(n_files, N_WINDOWS, -1)
    fp_f      = first_pass_flat.reshape(n_files, N_WINDOWS, -1)
    lab_f     = Y_full.reshape(n_files, N_WINDOWS, -1).astype(np.float32)
    fp_prob   = sigmoid(fp_f)
    residuals = lab_f - fp_prob

    n_val  = max(1, int(n_files * 0.15))
    rng    = torch.Generator(); rng.manual_seed(42)
    perm   = torch.randperm(n_files, generator=rng).numpy()
    val_i, train_i = perm[:n_val], perm[n_val:]

    emb_t  = torch.tensor(emb_f,     dtype=torch.float32)
    fp_t   = torch.tensor(fp_f,      dtype=torch.float32)
    res_t  = torch.tensor(residuals, dtype=torch.float32)
    site_t = torch.tensor(site_ids,  dtype=torch.long)
    hour_t = torch.tensor(hour_ids,  dtype=torch.long)

    model = ResidualSSM(n_classes=N_CLASSES,
                        d_model=rcfg["d_model"], d_state=rcfg["d_state"])
    print(f"  ResidualSSM params: {model.count_parameters():,}")

    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, epochs=n_epochs, steps_per_epoch=1, pct_start=0.1, anneal_strategy="cos")

    best_loss, best_state, wait = float("inf"), None, 0
    for ep in range(n_epochs):
        model.train()
        corr = model(emb_t[train_i], fp_t[train_i],
                     site_ids=site_t[train_i], hours=hour_t[train_i])
        loss = F.mse_loss(corr, res_t[train_i])
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        model.eval()
        with torch.no_grad():
            val_loss = F.mse_loss(
                model(emb_t[val_i], fp_t[val_i],
                      site_ids=site_t[val_i], hours=hour_t[val_i]),
                res_t[val_i])
        if val_loss.item() < best_loss:
            best_loss  = val_loss.item()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            if verbose: print(f"  Early stop ep {ep+1}")
            break

    model.load_state_dict(best_state)
    print(f"  ResidualSSM trained — best val MSE={best_loss:.6f}")
    return model, correction_weight


print("✅ ResidualSSM (zero-init head, MSE residual target) defined")

✅ ResidualSSM (zero-init head, MSE residual target) defined


In [16]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 14 — Test / Dry-Run inference (Perch pass on test files)
# ═══════════════════════════════════════════════════════════════════════
test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
IS_DRY_RUN = len(test_paths) == 0

if IS_DRY_RUN:
    n = CFG["dryrun_n_files"] or 20
    print(f"No hidden test — dry-run on {n} train files")
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:n]
else:
    print(f"Hidden test files: {len(test_paths)}")

meta_te, sc_te, emb_te = run_perch(test_paths, CFG["batch_files"], verbose=CFG["verbose"])
print(f"✅ Test Perch done — scores={sc_te.shape}  embs={emb_te.shape}")

No hidden test — dry-run on 20 train files
✅ Test Perch done — scores=(240, 234)  embs=(240, 1536)


In [17]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 15 — Full ProtoSSM + MLP + ResidualSSM pipeline
# ═══════════════════════════════════════════════════════════════════════
pcfg = CFG["proto"]
rcfg = CFG["residual"]
pp   = CFG["pp"]

# ── A: Helper: extract file-level site/hour tensors ───────────────────
def get_site_hour_tensors(meta_df, site2i, n_sites_cap=20):
    fnames   = meta_df.drop_duplicates("filename")["filename"].tolist()
    site_ids = np.array([
        min(site2i.get(meta_df.loc[meta_df["filename"]==fn,"site"].iloc[0], 0), n_sites_cap-1)
        for fn in fnames], dtype=np.int64)
    hour_ids = np.array([
        int(meta_df.loc[meta_df["filename"]==fn,"hour_utc"].iloc[0]) % 24
        for fn in fnames], dtype=np.int64)
    return (torch.tensor(site_ids, dtype=torch.long),
            torch.tensor(hour_ids, dtype=torch.long),
            site_ids, hour_ids)

# ── B: Train LightProtoSSM on all training data ───────────────────────
t0 = time.time()
print("Training LightProtoSSM…")
proto_model, site2i_tr = train_light_proto_ssm(
    emb_tr, sc_tr, Y_FULL_aligned, meta_tr,
    n_epochs=pcfg["n_epochs"], patience=pcfg["patience"],
    lr=pcfg["lr"], n_sites=pcfg["n_sites"], verbose=False)
print(f"ProtoSSM training: {time.time()-t0:.1f}s")

# ── C: ProtoSSM inference on TEST (no TTA here — TTA used for ResidualSSM training) ─
n_test_files = len(sc_te) // N_WINDOWS
emb_te_f     = emb_te.reshape(n_test_files, N_WINDOWS, -1)
sc_te_f      = sc_te.reshape(n_test_files, N_WINDOWS, -1)
site_te_t, hour_te_t, _, _ = get_site_hour_tensors(meta_te, site2i_tr)

proto_model.eval()
with torch.no_grad():
    proto_out = proto_model(
        torch.tensor(emb_te_f, dtype=torch.float32),
        torch.tensor(sc_te_f,  dtype=torch.float32),
        site_ids=site_te_t, hours=hour_te_t).numpy()
proto_scores_flat = proto_out.reshape(-1, N_CLASSES).astype(np.float32)

# ── D: Prior tables (3-tier: global → hour → site → site×hour) ────────
prior_tables   = build_prior_tables(sc, Y_SC)
sc_te_adjusted = apply_prior(
    sc_te, sites=meta_te["site"].to_numpy(),
    hours=meta_te["hour_utc"].to_numpy(),
    tables=prior_tables, lambda_prior=pp["prior_lambda"])

# ── E: MLP probes ──────────────────────────────────────────────────────
print("Training MLP probes…")
probe_models, emb_scaler, emb_pca, alpha_blend = train_mlp_probes(
    emb=emb_tr, scores_raw=sc_tr, Y=Y_FULL_aligned,
    min_pos=CFG["mlp"]["min_pos"], pca_dim=CFG["mlp"]["pca_dim"],
    alpha_blend=CFG["mlp"]["alpha_blend"])
sc_te_adjusted = apply_mlp_probes_vectorized(
    emb_te, sc_te_adjusted, probe_models, emb_scaler, emb_pca, alpha_blend)

# ── F: First-pass ensemble (50% ProtoSSM + 50% MLP-enhanced Perch) ────
ENSEMBLE_W      = 0.5
first_pass_flat = (ENSEMBLE_W * proto_scores_flat
                   + (1.0 - ENSEMBLE_W) * sc_te_adjusted)

# ── G: Compute training-set first-pass with TTA (for ResidualSSM) ─────
n_tr_files     = len(sc_tr) // N_WINDOWS
emb_tr_f       = emb_tr.reshape(n_tr_files, N_WINDOWS, -1)
sc_tr_f        = sc_tr.reshape(n_tr_files, N_WINDOWS, -1)
site_tr_t, hour_tr_t, tr_site_ids, tr_hour_ids = get_site_hour_tensors(
    meta_tr, site2i_tr)

proto_tr_out  = run_tta_proto(proto_model, emb_tr_f, sc_tr_f,
                               site_t=site_tr_t, hour_t=hour_tr_t)
proto_tr_flat = proto_tr_out.reshape(-1, N_CLASSES).astype(np.float32)

sc_tr_prior   = apply_prior(
    sc_tr, sites=meta_tr["site"].to_numpy(),
    hours=meta_tr["hour_utc"].to_numpy(),
    tables=prior_tables, lambda_prior=pp["prior_lambda"])
sc_tr_mlp     = apply_mlp_probes_vectorized(
    emb_tr, sc_tr_prior, probe_models, emb_scaler, emb_pca, alpha_blend)
first_pass_tr = ENSEMBLE_W * proto_tr_flat + (1.0 - ENSEMBLE_W) * sc_tr_mlp

# ── H: Isotonic calibration + per-class threshold optimization ─────────
print("Running isotonic calibration…")
train_probs_calib   = sigmoid(first_pass_tr)
PER_CLASS_THRESHOLDS = calibrate_and_optimize_thresholds(
    oof_probs=train_probs_calib, Y_FULL=Y_FULL_aligned, n_windows=N_WINDOWS)

# ── I: Train ResidualSSM ───────────────────────────────────────────────
t0 = time.time()
print("Training ResidualSSM…")
res_model, correction_weight = train_residual_ssm(
    emb_full=emb_tr, first_pass_flat=first_pass_tr, Y_full=Y_FULL_aligned,
    site_ids=tr_site_ids, hour_ids=tr_hour_ids,
    n_epochs=rcfg["n_epochs"], patience=rcfg["patience"],
    lr=rcfg["lr"], correction_weight=rcfg["correction_weight"])
print(f"ResidualSSM training: {time.time()-t0:.1f}s")

# ── J: Apply ResidualSSM correction to TEST ────────────────────────────
first_pass_te_f = first_pass_flat.reshape(n_test_files, N_WINDOWS, -1)
res_model.eval()
with torch.no_grad():
    test_correction = res_model(
        torch.tensor(emb_te_f,        dtype=torch.float32),
        torch.tensor(first_pass_te_f, dtype=torch.float32),
        site_ids=site_te_t, hours=hour_te_t).numpy()
correction_flat = test_correction.reshape(-1, N_CLASSES).astype(np.float32)
final_scores    = first_pass_flat + correction_weight * correction_flat

# ── K: Temperature scaling → sigmoid → full post-processing chain ─────
final_scores = final_scores / temperatures[None, :]
probs        = sigmoid(final_scores)
probs        = file_confidence_scale(probs, power=pp["conf_scale_power"],
                                      top_k=pp["conf_scale_top_k"])
probs        = rank_aware_scaling(probs, power=pp["rank_scale_power"])
probs        = adaptive_delta_smooth(probs, base_alpha=pp["delta_smooth_alpha"])
probs        = apply_per_class_thresholds(probs, PER_CLASS_THRESHOLDS)  # RE-ENABLED
probs        = np.clip(probs, 0.0, 1.0)

# ── L: Save ProtoSSM submission ────────────────────────────────────────
sub_proto = pd.DataFrame(probs.astype(np.float32), columns=PRIMARY_LABELS)
sub_proto.insert(0, "row_id", meta_te["row_id"].values)
assert list(sub_proto.columns) == ["row_id"] + PRIMARY_LABELS
assert not sub_proto.isna().any().any()
sub_proto.to_csv("submission_protossm.csv", index=False)
print(f"✅ submission_protossm.csv saved — shape {sub_proto.shape}")

# Free memory before SED
del emb_tr_f, sc_tr_f, proto_model, res_model, proto_tr_out
gc.collect()
print(f"Wall time so far: {(time.time() - _WALL_START)/60:.1f} min")

Training LightProtoSSM…
  LightProtoSSM params: 723,093
  ProtoSSM trained — best loss=0.9880
ProtoSSM training: 13.3s
Training MLP probes…
  PCA: (708, 1536) → (708, 64)  var=81.47%


MLP probes:   0%|          | 0/58 [00:00<?, ?it/s]

  Trained 58 MLP probes
Running isotonic calibration…
Calibrated 31/234 classes | mean_thr=0.462 range=[0.20, 0.50]
Training ResidualSSM…
  ResidualSSM params: 176,010
  ResidualSSM trained — best val MSE=0.023711
ResidualSSM training: 1.9s
✅ submission_protossm.csv saved — shape (240, 235)
Wall time so far: 3.8 min


In [18]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 16 — Tucker Arrants distilled SED ONNX inference (5 folds)
# ═══════════════════════════════════════════════════════════════════════
import librosa
from scipy.ndimage import gaussian_filter1d

N_MELS_SED = 256; N_FFT_SED = 2048; HOP_SED = 512
FMIN_SED = 20;    FMAX_SED = 16000; TOP_DB_SED = 80

def find_sed_dir():
    hits = sorted(Path("/kaggle/input").rglob("sed_fold0.onnx"))
    if not hits:
        raise FileNotFoundError("sed_fold0.onnx not found. Attach tuckerarrants/bc2026-distilled-sed-public.")
    return hits[0].parent

def make_sed_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 4
    so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), sess_options=so, providers=["CPUExecutionProvider"])

def audio_to_mel_256(chunks):
    mels = []
    for x in chunks:
        s = librosa.feature.melspectrogram(
            y=x, sr=SR, n_fft=N_FFT_SED, hop_length=HOP_SED,
            n_mels=N_MELS_SED, fmin=FMIN_SED, fmax=FMAX_SED, power=2.0)
        s = librosa.power_to_db(s, top_db=TOP_DB_SED)
        s = (s - s.mean()) / (s.std() + 1e-6)
        mels.append(s)
    return np.stack(mels)[:, None].astype(np.float32)

def file_to_chunks(path):
    y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if sr0 != SR: y = librosa.resample(y, orig_sr=sr0, target_sr=SR)
    n = 60 * SR
    if len(y) < n: y = np.pad(y, (0, n - len(y)))
    else: y = y[:n]
    return y.reshape(N_WINDOWS, WINDOW_SAMPLES), np.arange(1, N_WINDOWS+1) * WINDOW_SEC

def sigmoid_np(x):
    return (1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))).astype(np.float32)

sed_dir          = find_sed_dir()
sed_fold_paths   = sorted(sed_dir.glob("sed_fold*.onnx"),
                           key=lambda p: int(re.search(r"sed_fold(\d+)", p.name).group(1)))
sed_sessions     = [make_sed_session(p) for p in sed_fold_paths]
print(f"SED folds loaded: {[p.name for p in sed_fold_paths]}")

sed_rows, sed_preds = [], []
_t0_sed = time.time()

for i, path in enumerate(test_paths, 1):
    chunks, ends = file_to_chunks(path)
    mel   = audio_to_mel_256(chunks)
    p_sum = np.zeros((len(chunks), N_CLASSES), dtype=np.float32)
    for sess in sed_sessions:
        outs = sess.run(None, {sess.get_inputs()[0].name: mel})
        p_sum += 0.5 * sigmoid_np(outs[0]) + 0.5 * sigmoid_np(outs[1].max(axis=1))
    p_mean = p_sum / len(sed_sessions)
    if len(p_mean) > 1:
        p_mean = gaussian_filter1d(p_mean, sigma=0.65, axis=0, mode="nearest").astype(np.float32)
    stem = path.stem
    sed_rows.extend([f"{stem}_{int(t)}" for t in ends])
    sed_preds.append(p_mean)
    if i % 50 == 0 or i == len(test_paths):
        print(f"  SED: {i}/{len(test_paths)} | {time.time()-_t0_sed:.1f}s")

sed_preds_arr = np.concatenate(sed_preds, axis=0)
sed_sub = pd.DataFrame(np.clip(sed_preds_arr, 0.0, 1.0), columns=PRIMARY_LABELS)
sed_sub.insert(0, "row_id", sed_rows)
sed_sub.to_csv("submission_sed.csv", index=False)
print(f"✅ submission_sed.csv saved: {sed_sub.shape}  time={time.time()-_t0_sed:.1f}s")

SED folds loaded: ['sed_fold0.onnx', 'sed_fold1.onnx', 'sed_fold2.onnx', 'sed_fold3.onnx', 'sed_fold4.onnx']
  SED: 20/20 | 69.2s
✅ submission_sed.csv saved: (240, 235)  time=69.3s


In [19]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 17 — v5 SED trio inference (mel-128, cluster/focal/pseudo)
#            From NB3 (perch-sed-lb-0-946-clap), Cell 11b
# ═══════════════════════════════════════════════════════════════════════
_V5_DIR = None
for _vd in ["/kaggle/input/datasets/needless090/birdclef2026-sed-v5-trio",
            "/kaggle/input/birdclef2026-sed-v5-trio"]:
    if os.path.exists(_vd): _V5_DIR = _vd; break
print(f"[v5] SED dir: {_V5_DIR}")

V5_NAMES    = ["v5_cluster_aware.onnx", "v5_focal.onnx", "v5_pseudo2.onnx",
               "v5_pseudo.onnx", "v5_external.onnx"]
v5_sessions = []
if _V5_DIR:
    _so_v5 = ort.SessionOptions()
    _so_v5.intra_op_num_threads = 4
    _so_v5.inter_op_num_threads = 1
    _so_v5.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    for nm in V5_NAMES:
        p = f"{_V5_DIR}/{nm}"
        if os.path.exists(p):
            v5_sessions.append((nm, ort.InferenceSession(p, sess_options=_so_v5,
                                                          providers=["CPUExecutionProvider"])))
            print(f"  v5 loaded: {nm}")

def mel_128(y_chunk):
    S  = librosa.feature.melspectrogram(y=y_chunk, sr=SR, n_fft=2048, hop_length=512,
                                          n_mels=128, fmin=20, fmax=16000)
    Sd = librosa.power_to_db(S, ref=np.max)
    Sd = (Sd + 80.0) / 80.0
    return np.clip(Sd, 0, 1)[None, :, :].astype(np.float32)

v5_rows, v5_preds_list = [], []
_t0_v5 = time.time()

if v5_sessions:
    for i, path in enumerate(test_paths, 1):
        chunks, ends = file_to_chunks(path)
        vb = np.stack([mel_128(c.astype(np.float32)) for c in chunks], axis=0)
        all_v5 = []
        for _, s in v5_sessions:
            logits = s.run(None, {s.get_inputs()[0].name: vb})[0]
            all_v5.append(sigmoid_np(logits))
        v5_mean = np.mean(np.stack(all_v5), axis=0).astype(np.float32)
        stem = path.stem
        v5_rows.extend([f"{stem}_{int(t)}" for t in ends])
        v5_preds_list.append(v5_mean)
        if i % 50 == 0 or i == len(test_paths):
            print(f"  v5 SED: {i}/{len(test_paths)} | {time.time()-_t0_v5:.1f}s")
    v5_preds_arr = np.concatenate(v5_preds_list, axis=0)
    v5_sub = pd.DataFrame(np.clip(v5_preds_arr, 0.0, 1.0), columns=PRIMARY_LABELS)
    v5_sub.insert(0, "row_id", v5_rows)
    v5_sub.to_csv("submission_v5.csv", index=False)
    print(f"✅ submission_v5.csv saved: {v5_sub.shape}  time={time.time()-_t0_v5:.1f}s")
else:
    print("⚠️  v5 SED not available — skipping (3-way blend will be used)")

[v5] SED dir: None
⚠️  v5 SED not available — skipping (3-way blend will be used)


In [20]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 18 — CLAP 5th stream (budget-aware, dynamic abort)
#            From NB3 (perch-sed-lb-0-946-clap), Cell 11d
# ═══════════════════════════════════════════════════════════════════════
_clap_dir = None
for _cd in ["/kaggle/input/datasets/needless090/birdclef2026-clap-probe",
            "/kaggle/input/birdclef2026-clap-probe"]:
    if os.path.exists(_cd): _clap_dir = _cd; break

clap_preds        = None
clap_per_row_mask = None

if _clap_dir is not None:
    try:
        from transformers import ClapModel, ClapProcessor
        _CSR      = 48000
        _BUDGET   = 45 * 60   # 45-min CLAP budget
        _cproc    = ClapProcessor.from_pretrained(_clap_dir)
        _cmod     = ClapModel.from_pretrained(_clap_dir).eval()
        torch.set_num_threads(4)
        _W     = np.load(f"{_clap_dir}/clap_probe_W.npy")
        _b     = np.load(f"{_clap_dir}/clap_probe_b.npy")
        _fmask = np.load(f"{_clap_dir}/clap_probe_fitmask.npy")
        print(f"[CLAP] probe loaded: {_fmask.sum()} fit species")

        _N_TEST     = len(test_paths)
        clap_preds  = np.zeros((_N_TEST * N_WINDOWS, N_CLASSES), dtype=np.float32)
        _CST        = time.time()
        _aborted_at = None

        with torch.inference_mode():
            for _fi, _fp in enumerate(test_paths):
                _el = time.time() - _CST
                if _el > _BUDGET:
                    _aborted_at = _fi; break
                if _fi >= 30 and (_el / _fi) * _N_TEST > _BUDGET:
                    _aborted_at = _fi; break
                _y, _sr0 = sf.read(str(_fp), dtype="float32", always_2d=False)
                if _y.ndim == 2: _y = _y.mean(axis=1)
                if _sr0 != SR: _y = librosa.resample(_y, orig_sr=_sr0, target_sr=SR)
                _n = 60 * SR
                if len(_y) < _n: _y = np.pad(_y, (0, _n - len(_y)))
                else: _y = _y[:_n]
                segs = [_y[wi*WINDOW_SAMPLES:(wi+1)*WINDOW_SAMPLES].astype(np.float32)
                        for wi in range(N_WINDOWS)]
                segs = [np.pad(s, (0, max(0, WINDOW_SAMPLES - len(s)))) for s in segs]
                inp   = _cproc(audios=segs, sampling_rate=_CSR, return_tensors="pt", padding=True)
                embs  = _cmod.get_audio_features(**inp).cpu().numpy().astype(np.float32)
                embs /= (np.linalg.norm(embs, axis=1, keepdims=True) + 1e-9)
                logit = embs @ _W + _b
                clap_preds[_fi*N_WINDOWS:(_fi+1)*N_WINDOWS] = sigmoid_np(logit)
                if (_fi+1) % 50 == 0:
                    print(f"  CLAP {_fi+1}/{_N_TEST}  elapsed={time.time()-_CST:.0f}s")

        _mk = np.zeros(_N_TEST, dtype=bool)
        if _aborted_at is None:
            _mk[:] = True
            print(f"[CLAP] done all {_N_TEST} files ({time.time()-_CST:.1f}s)")
        else:
            _mk[:_aborted_at] = True
            print(f"[CLAP] partial: {_aborted_at}/{_N_TEST} ({time.time()-_CST:.1f}s)")

        if _mk.any():
            _cl_sub = pd.DataFrame(np.clip(clap_preds, 0.0, 1.0), columns=PRIMARY_LABELS)
            _row_ids = []
            for _fp in test_paths:
                stem = _fp.stem
                for t_w in (np.arange(1, N_WINDOWS+1) * WINDOW_SEC):
                    _row_ids.append(f"{stem}_{int(t_w)}")
            _cl_sub.insert(0, "row_id", _row_ids)
            _cl_sub.to_csv("submission_clap.csv", index=False)
            np.save("clap_filemask.npy", _mk)
            clap_per_row_mask = np.repeat(_mk, N_WINDOWS)
            print(f"✅ submission_clap.csv saved (mask {_mk.sum()}/{len(_mk)} files)")

        del _cmod, _cproc; gc.collect()
    except Exception as _ce:
        import traceback; traceback.print_exc()
        print(f"[CLAP] FAILED: {_ce} — falling back to 4-way blend")
        clap_preds = None; clap_per_row_mask = None
else:
    print("[CLAP] Dataset not found — skipping (4-way or 3-way blend)")

[CLAP] Dataset not found — skipping (4-way or 3-way blend)


In [21]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 19 — Head weights (linear probe on train-audio embeddings)
#            From NB2 (gate-fake008-head0015)
# ═══════════════════════════════════════════════════════════════════════
import glob as _glob

head_rank = None
HEAD_RANK_BLEND = 0.05

head_candidates = _glob.glob("/kaggle/input/**/head_weights_train_audio.npz", recursive=True)
if head_candidates:
    try:
        hw     = np.load(head_candidates[0], allow_pickle=True)
        head_W = hw["W"].astype(np.float32)
        head_b = hw["b"].astype(np.float32)
        head_mask = hw["trained_mask"].astype(bool).reshape(1, -1)
        assert emb_te.shape[1] == head_W.shape[1], f"Dim mismatch: emb={emb_te.shape[1]} W={head_W.shape[1]}"
        head_logits = emb_te.astype(np.float32) @ head_W.T + head_b
        head_logits = head_logits * head_mask.astype(np.float32)
        head_probs  = sigmoid(head_logits)
        EPS         = 1e-5
        head_rank   = pd.DataFrame(np.clip(head_probs, EPS, 1.0 - EPS)).rank(
            axis=0, pct=True).to_numpy(np.float32)
        print(f"✅ Head weights loaded: trained on {int(head_mask.sum())} species")
    except Exception as e:
        print(f"⚠️  Head weights load failed: {e}")
        head_rank = None
else:
    print("[Head] head_weights_train_audio.npz not found — skipping")

[Head] head_weights_train_audio.npz not found — skipping


In [22]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 20 — Final rank ensemble with fat-tail gates
#            Best of NB2 (gate-fake008) + NB3 (v222 5-way) combined
# ═══════════════════════════════════════════════════════════════════════
EPS = 1e-5
bl  = CFG["blend"]

# ── Load all branches ──────────────────────────────────────────────────
df_a = pd.read_csv("submission_protossm.csv")
df_b = pd.read_csv("submission_sed.csv")
cols = [c for c in df_a.columns if c != "row_id"]
df_b = df_b.set_index("row_id").loc[df_a["row_id"]].reset_index()

pa = np.clip(df_a[cols].to_numpy(np.float32), EPS, 1 - EPS)
pb = np.clip(df_b[cols].to_numpy(np.float32), EPS, 1 - EPS)

row_ids  = df_a["row_id"].astype(str).to_numpy()
file_ids = np.array(["_".join(r.split("_")[:-1]) for r in row_ids])

xa = pd.DataFrame(pa).rank(axis=0, pct=True).to_numpy(np.float32)
xb = pd.DataFrame(pb).rank(axis=0, pct=True).to_numpy(np.float32)

# Optional streams
xv = xc = None
if os.path.exists("submission_v5.csv"):
    _v5 = pd.read_csv("submission_v5.csv")
    if len(_v5) == len(df_a):
        _v5 = _v5.set_index("row_id").loc[df_a["row_id"]].reset_index()
        xv  = pd.DataFrame(np.clip(_v5[cols].to_numpy(np.float32), EPS, 1-EPS)).rank(
            axis=0, pct=True).to_numpy(np.float32)
        print(f"v5 SED loaded ({len(_v5)} rows)")

if os.path.exists("submission_clap.csv") and os.path.exists("clap_filemask.npy"):
    _cl = pd.read_csv("submission_clap.csv")
    _mk = np.load("clap_filemask.npy")
    if len(_cl) == len(df_a):
        _cl = _cl.set_index("row_id").loc[df_a["row_id"]].reset_index()
        xc  = pd.DataFrame(np.clip(_cl[cols].to_numpy(np.float32), EPS, 1-EPS)).rank(
            axis=0, pct=True).to_numpy(np.float32)
        clap_per_row_mask = np.repeat(_mk, N_WINDOWS)
        print(f"CLAP stream loaded, mask={_mk.sum()}/{len(_mk)} files")

# ── Build base blend (2/3/4/5-way depending on available streams) ──────
SED_W, V5_W, CLAP_W = bl["sed_w"], bl["v5_w"], bl["clap_w"]

if xv is not None:
    base_w = 1.0 - SED_W - V5_W
    base   = xa * base_w + xb * SED_W + xv * V5_W
    if xc is not None and clap_per_row_mask is not None and clap_per_row_mask.any():
        new_pa  = 1.0 - SED_W - V5_W - CLAP_W
        five_way = xa * new_pa + xb * SED_W + xv * V5_W + xc * CLAP_W
        pred     = np.where(clap_per_row_mask[:, None], five_way, base)
        print(f"5-way blend for {clap_per_row_mask.sum()} rows; 3-way for rest")
    else:
        pred = base
        print("3-way blend (ProtoSSM + SED + v5)")
else:
    pred = xa * (1.0 - SED_W) + xb * SED_W
    print("2-way fallback blend (ProtoSSM + SED)")

# ── Optionally inject head-weights branch ─────────────────────────────
if head_rank is not None:
    pred = (1.0 - HEAD_RANK_BLEND) * pred + HEAD_RANK_BLEND * head_rank
    print(f"Head-weights blended in (+{HEAD_RANK_BLEND:.2f})")

# ── Gate 1: Fake-only noise suppression ───────────────────────────────
fake_only = (pa > bl["fake_only_thr"]) & (pb < bl["sed_low_thr"])
pred = np.where(fake_only,
                (1.0 - bl["fake_blend"]) * pred + bl["fake_blend"] * xa,
                pred)

# ── Gate 2: Fat-tail temporal continuity rescue ───────────────────────
R   = bl["cont_radius"]
offs = np.arange(-R, R+1, dtype=np.float32)
proto_kernel = (1.0 + (offs / bl["cont_scale"]) ** 2 / bl["cont_df"]) ** (-(bl["cont_df"] + 1.0) / 2.0)
proto_kernel = (proto_kernel / proto_kernel.sum()).astype(np.float32)

pa_ctx = pa.copy()
for fid in pd.unique(file_ids):
    m = file_ids == fid
    x = pa[m]
    if len(x) > 1:
        xp = np.pad(x, ((R, R), (0, 0)), mode="edge")
        pa_ctx[m] = sum(proto_kernel[i] * xp[i:i+len(x)] for i in range(2*R+1))

xctx       = pd.DataFrame(pa_ctx).rank(axis=0, pct=True).to_numpy(np.float32)
proto_cont = ((xctx > bl["cont_rank_thr"]) & (xa > bl["cont_local_thr"])
              & (pb < bl["sed_cont_low"]) & ~fake_only)
pred = np.where(proto_cont,
                (1.0 - bl["cont_blend"]) * pred + bl["cont_blend"] * np.maximum(xa, xctx),
                pred)

# ── Gate 3: Rare SED spike rescue ─────────────────────────────────────
sed_only = ((xb > bl["sed_only_thr"]) & (xa < bl["fake_rank_low"])
            & ~fake_only & ~proto_cont)
pred = np.where(sed_only,
                (1.0 - bl["sed_only_blend"]) * pred + bl["sed_only_blend"] * xb,
                pred)

print(f"Gates applied — fake_only: {fake_only.sum()} | proto_cont: {proto_cont.sum()} | sed_only: {sed_only.sum()}")

# ── Sonotype mirroring (Insecta visual-form groups) ────────────────────
sub = df_a.copy()
sub[cols] = pred.astype(np.float32)

MIRROR_PAIRS = (
    ("47158son15", "47158son16"),
    ("47158son09", "47158son12"),
    ("47158son02", "47158son14"),
    ("47158son13", "47158son21", "47158son22", "47158son23"),
)
col_to_idx = {l: i for i, l in enumerate(cols)}
mirror_count = 0
for group in MIRROR_PAIRS:
    valid_idx = [col_to_idx[s] for s in group if s in col_to_idx]
    if len(valid_idx) >= 2:
        group_max = sub[cols].iloc[:, valid_idx].max(axis=1).to_numpy(np.float32)
        for idx in valid_idx:
            sub.iloc[:, idx + 1] = group_max
        mirror_count += len(valid_idx)
print(f"Sonotype mirroring: {mirror_count} columns")

# ── Adaptive thresholding (suppress noisy rare-taxa predictions) ───────
try:
    _tax = pd.read_csv(BASE / "taxonomy.csv").set_index("primary_label")
    RARE_CLASSES = {"Amphibia", "Mammalia", "Reptilia"}
    rare_count   = 0
    for ci, sp in enumerate(cols):
        if sp in _tax.index and _tax.loc[sp, "class_name"] in RARE_CLASSES:
            col_idx = ci + 1
            vals    = sub.iloc[:, col_idx].to_numpy(np.float32)
            thr     = vals.mean() + 0.05
            sub.iloc[:, col_idx] = np.where(vals < thr, vals * 0.9, vals)
            rare_count += 1
    print(f"Adaptive thresholding: {rare_count} rare species suppressed")
except Exception as e:
    print(f"Adaptive thresholding skipped: {e}")

# ── Dry-run alignment ──────────────────────────────────────────────────
OUT_CSV = "submission.csv"
if IS_DRY_RUN:
    print("Dry-run: aligning to sample_submission.csv")
    sample_public = pd.read_csv(BASE / "sample_submission.csv")
    template      = sub[cols].mean(axis=0).astype(np.float32)
    sub           = sample_public.copy()
    for label in cols:
        sub[label] = template[label]

sub.to_csv(OUT_CSV, index=False)
print(f"✅ submission.csv saved — shape {sub.shape}")
print(f"Total wall time: {(time.time() - _WALL_START)/60:.1f} min")

2-way fallback blend (ProtoSSM + SED)
Gates applied — fake_only: 2396 | proto_cont: 4690 | sed_only: 805
Sonotype mirroring: 10 columns
Adaptive thresholding: 44 rare species suppressed
Dry-run: aligning to sample_submission.csv
✅ submission.csv saved — shape (3, 235)
Total wall time: 6.1 min
